In [ ]:
import os
import sys
import itertools as itr
sys.path.append("../")
sys.path.append("../..")
sys.path.append("../../baselines/uai2022-intervention-estimation-latents/main_codes")
sys.path.append("../../src")
import os
import pickle

import numpy as np
from causaldag import unknown_target_igsp
from causaldag import partial_correlation_test, MemoizedCI_Tester, partial_correlation_suffstat
from causaldag import MemoizedInvarianceTester, gauss_invariance_test, gauss_invariance_suffstat
from src.tools.metric import get_compared_components, get_skeleton, metric_skeleton_level, metric_cpdag_level

from run_sachs_data import IMAG_sachs

In [ ]:
def metric_target_level_for_utigsp(pred, targ):
    """ P / R / F1 (I-TARGET Level) """
    TP , TP_FP, TP_FN = 0, 0, 0
    for set_pred, set_targ in zip(pred, targ):
        TP += len(set_pred.intersection(set_targ))
        TP_FP += len(set_pred)
        TP_FN += len(set_targ)
    precision = TP / max(TP_FP, 1)
    recall = TP / max(TP_FN, 1)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall != 0 else 0
    pred_iden_edges_num, targ_iden_edges_num = TP_FP, TP_FN
    return {'p':round(precision,2), 'r':round(recall,2), 'f1':round(f1,2), '#pred_iden_edges':int(pred_iden_edges_num), '#targ_iden_edges':int(targ_iden_edges_num)}

In [ ]:
exp_name = 'exp_200'
intervention_size_list = [1, 1, 2, 2, 4, 5, 6, 7, 7, 10, 11, 14, 15, 22]
os.makedirs(f'../../exps_of_result/ut-igsp/{exp_name}/know', exist_ok=True)
os.makedirs(f'../../baselines/exps_of_result/ut-igsp/{exp_name}/unknow', exist_ok=True)


for idx, benchmark_name in enumerate(['01earthquake', '02survey', '03asia', '04sachs',  '05child', '06insurance', '07water', '08mildew', '09alarm', '10barley', '11hailfinder', '12hepar2', '13win95pts', '14pathfinder']):
    if idx<=4:
        continue
    print(f"{'-'*60} {benchmark_name} {'-'*60} \n")
    intervention_size = intervention_size_list[idx]
    # ground truth of I-SKELETON / I-TARGETS / I-CPDAG
    aug_graph_path = f'../../datasets/experiment/original/{exp_name}/aug_graphs/{benchmark_name}_aug_graph.txt'
    raw_dataset_path = f'../../datasets/experiment/original/{exp_name}/raw_samples/{benchmark_name}_raw_dataset.pkl'
    int_targets_path = f'../../datasets/experiment/original/{exp_name}/int_targets/{benchmark_name}_int_targets.pkl'
    targ_I_SKELETON, targ_I_TARGETS, targ_I_CPDAG = get_compared_components(aug_graph_path, intervention_size, real=True)
    
    try:
        know_pred_graph_path = f'../../baselines/exps_of_result/ut-igsp/{exp_name}/know/{benchmark_name}_aug_graph.txt'
        unknow_pred_graph_path = f'../../baselines/exps_of_result/ut-igsp/{exp_name}/unknow/{benchmark_name}_aug_graph.txt'

        with open(raw_dataset_path, 'rb') as f:
            data_list = pickle.load(f)
        with open(int_targets_path, 'rb') as f:
            know_targets_list = pickle.load(f)
        
        obs_samples, iv_samples_list = data_list[0], data_list[1:]
        
        nodes = set(range(obs_samples.shape[1]))
        obs_suffstat = partial_correlation_suffstat(obs_samples)
        ci_tester = MemoizedCI_Tester(partial_correlation_test, obs_suffstat, alpha=1e-3)
        invariance_suffstat = gauss_invariance_suffstat(obs_samples, iv_samples_list)
        invariance_tester = MemoizedInvarianceTester(gauss_invariance_test, invariance_suffstat, alpha=1e-3)
        
        know_setting_list = [dict(known_interventions=frozenset(targets)) for idx, targets in enumerate(know_targets_list[1:])]
            
        S_obs = (obs_samples.T@obs_samples)/obs_samples.shape[0]
        S_all = {}
        # add the observational to the mix
        S_all['setting_0'] = S_obs

        for idx_setting in range(len(know_setting_list)):
            S_current = (iv_samples_list[idx_setting].T@iv_samples_list[idx_setting])/iv_samples_list[idx_setting].shape[0]
            S_all['setting_%d'%(idx_setting+1)] = S_current
        
        # for S_Delta estimation
        lambda_1_list = [0.3]
        # other PDEs
        lambda_pasp_list = [0.2]
        # remove small values
        th1_list = [0.01]
        rho = 1.0
        n_max_iter = 500
        stop_cond = 1e-6
        tol = 1e-9
        verbose = False
        only_diag = False
        return_pasp = True
        max_subset_size = None

        parameters_list = \
            list(itr.product(lambda_1_list,lambda_pasp_list,th1_list))
        
        save_min = 0
        save_mt = {}
        for parameters in parameters_list:
            est_edges, est_skeleton, K_hat_all, K_pasp_hat_all ,S_Delta_size_all, time_all = IMAG_sachs(S_obs, S_all,max_subset_size,\
                parameters[0],parameters[1],rho,parameters[2],n_max_iter,stop_cond,tol,verbose,only_diag,return_pasp)
            pred_I_TARGETS_unknow = [set(value) for key, value in K_hat_all.items()]
        
            mt_target_unknow = metric_target_level_for_utigsp(pred_I_TARGETS_unknow, list(map(set, know_targets_list[1:])))
            if mt_target_unknow['f1'] > save_min:
                save_min = mt_target_unknow['f1']
                save_mt = mt_target_unknow
        print(f'unknow performance: \n targets:{save_mt} \n')

    except:
        print(f'pass {benchmark_name}\n')